In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython import get_ipython
import tkinter as tk
from tkinter import filedialog
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px # For accessing built-in color scales
from scipy.interpolate import griddata
from plotly.subplots import make_subplots
import imageio
try:
    import imageio
    IMAGEIO_AVAILABLE = True
except ImportError:
    IMAGEIO_AVAILABLE = False
    print("⚠️ Warning: 'imageio' library not found. GIF export will be disabled.")
    print("   To enable GIF export, run '!pip install imageio' in a new notebook cell and restart the kernel.")

pio.renderers.default = "notebook"

def _parse_timestamp(series):
    # (This function remains unchanged)
    ts = pd.to_datetime(series, format="%d.%m.%Y %H:%M:%S.%f", errors="coerce")
    if ts.notna().any(): return ts
    return pd.to_datetime(series, dayfirst=True, errors="coerce")

print("Libraries and helpers loaded. Plotly renderer and color scales configured.")

In [ ]:
def _generate_time_axis(num_points=1200, duration_seconds=30):
    """Helper to create a smooth time axis for test data. Increased points for smoother data."""
    time_deltas = pd.to_timedelta(np.arange(num_points) * (duration_seconds / num_points), unit='s')
    start_time = pd.Timestamp.now().floor('s')
    return start_time + time_deltas

# --- Test Scenario 1: The Fast Center-Out Ripple (Corrected) ---
def generate_center_out_ripple_data(config):
    print("🔬 TEST MODE: Generating 'Fast Center-Out Ripple' data...")
    spatial_map, all_columns, control_col = config["spatial_map"], config["requested_columns"], "70418_T901000_W1T"
    timestamps = _generate_time_axis() # Using the corrected helper
    center_pos = np.array([0.5, 0.5])
    df = pd.DataFrame(index=timestamps); df['datetime'] = df.index
    base_temp, intensity = 800.0, 25.0
    for ts in timestamps:
        progress = (ts - timestamps[0]) / (timestamps[-1] - timestamps[0])
        for col in all_columns:
            temp = base_temp
            if col in spatial_map:
                sensor_pos = np.array(spatial_map[col])
                distance = np.linalg.norm(sensor_pos - center_pos)
                effect = np.sin(distance * 30 - progress * 40) * intensity
                temp += effect
            if col == control_col: temp = base_temp
            df.loc[ts, col] = temp
    df = df.reset_index(drop=True); df['source_file'] = "Center-Out Ripple Test"
    print("✅ Test data generated successfully.")
    return df

# --- Test Scenario 2: The Interfering Waves (Corrected) ---
def generate_interfering_waves_data(config):
    print("🔬 TEST MODE: Generating 'Interfering Waves' data...")
    spatial_map, all_columns, control_col = config["spatial_map"], config["requested_columns"], "70418_T901000_W1T"
    timestamps = _generate_time_axis() # Using the corrected helper
    source1_pos, source2_pos = np.array([0.2, 0.75]), np.array([0.8, 0.25])
    df = pd.DataFrame(index=timestamps); df['datetime'] = df.index
    base_temp, intensity1, intensity2 = 800.0, 20.0, -20.0
    for ts in timestamps:
        progress = (ts - timestamps[0]) / (timestamps[-1] - timestamps[0])
        for col in all_columns:
            temp = base_temp
            if col in spatial_map:
                sensor_pos = np.array(spatial_map[col])
                dist1 = np.linalg.norm(sensor_pos - source1_pos)
                dist2 = np.linalg.norm(sensor_pos - source2_pos)
                effect1 = np.sin(dist1 * 25 - progress * 50) * intensity1
                effect2 = np.sin(dist2 * 35 - progress * 50) * intensity2
                temp += effect1 + effect2
            if col == control_col: temp = base_temp
            df.loc[ts, col] = temp
    df = df.reset_index(drop=True); df['source_file'] = "Interfering Waves Test"
    print("✅ Test data generated successfully.")
    return df

# --- Test Scenario 3: The Pulsing Hotspot (Corrected) ---
def generate_pulsing_hotspot_data(config):
    print("🔬 TEST MODE: Generating 'Pulsing Hotspot' data...")
    spatial_map, all_columns, control_col = config["spatial_map"], config["requested_columns"], "70418_T901000_W1T"
    timestamps = _generate_time_axis() # Using the corrected helper
    hotspot_pos = np.array([0.7, 0.3])
    df = pd.DataFrame(index=timestamps); df['datetime'] = df.index
    base_temp, max_intensity = 800.0, 40.0
    for ts in timestamps:
        progress = (ts - timestamps[0]) / (timestamps[-1] - timestamps[0])
        pulse = (np.sin(progress * 20 * (1 + progress * 3)) + 1) / 2 
        current_intensity = max_intensity * pulse
        for col in all_columns:
            temp = base_temp
            if col in spatial_map:
                sensor_pos = np.array(spatial_map[col])
                distance = np.linalg.norm(sensor_pos - hotspot_pos)
                effect = np.exp(-(distance**2) / 0.05) * current_intensity
                temp += effect
            if col == control_col: temp = base_temp
            df.loc[ts, col] = temp
    df = df.reset_index(drop=True); df['source_file'] = "Pulsing Hotspot Test"
    print("✅ Test data generated successfully.")
    return df

print("✅ Advanced (and much faster) test data generator suite defined.")

In [ ]:
class ThermocoupleAnalyzer:
    """
    Final version: An object-oriented class that loads, analyzes, and plots data,
    with a user-controlled, aesthetically perfect, animated contour plot and a
    final comparison to a reference benchmark.
    """
    def __init__(self, file_paths, config, imageio_available=False):
        if file_paths is not None and not isinstance(file_paths, (list, tuple)):
            file_paths = [file_paths]
        self.file_paths = file_paths
        self.config = config
        self.imageio_available = imageio_available
        self.data = None
        self.available_columns, self.missing_columns, self.color_map = [], [], {}
        self.colors = self.config.get("custom_colors") or px.colors.qualitative.D3
        self.figures_to_save = {}

    def load_data(self):
        all_dataframes, all_found_columns = [], set()
        for file_path in self.file_paths:
            print(f"--- Processing file: {os.path.basename(file_path)} ---")
            try:
                df_peek = pd.read_excel(file_path, header=None, nrows=60, dtype=str) if file_path.lower().endswith((".xlsx", ".xls")) else pd.read_csv(file_path, sep=';', header=None, nrows=60, dtype=str, engine='python')
                header_idx = next((i for i, r in enumerate(df_peek.itertuples(index=False)) if any("TimeStamp" in str(v) for v in r)), None)
                if header_idx is None: print("  -> 'TimeStamp' header not found. Skipping."); continue
                skip_rows, header_row = header_idx + 1, df_peek.iloc[header_idx].fillna("").astype(str).str.strip().tolist()
                df = pd.read_excel(file_path, header=None, skiprows=skip_rows, dtype=str) if file_path.lower().endswith((".xlsx", ".xls")) else pd.read_csv(file_path, sep=';', header=None, skiprows=skip_rows, dtype=str, engine='python')
                while header_row and not header_row[-1]: header_row.pop()
                df = df.iloc[:, :len(header_row)]; df.columns = header_row; df = df.dropna(axis=0, how="all").reset_index(drop=True)
                if df.empty: continue
                ts_col, t_col = header_row[0], (header_row[1] if len(header_row) > 1 else None)
                df["datetime"] = _parse_timestamp(df[ts_col].astype(str).str.strip() + " " + df[t_col].astype(str).str.strip()) if t_col and df[t_col].astype(str).str.contains(":").any() else _parse_timestamp(df[ts_col].astype(str).str.strip())
                df = df.dropna(subset=["datetime"])
                for col in [c for c in df.columns if c not in ["datetime", ts_col, t_col]]: df[col] = pd.to_numeric(df[col].astype(str).str.replace(",", ".", regex=False).str.strip(), errors="coerce")
                df['source_file'] = os.path.basename(file_path); all_dataframes.append(df); all_found_columns.update(df.columns)
            except Exception as e: print(f"  -> Could not read file. Skipping. Error: {e}")
        if not all_dataframes: raise ValueError("Could not load any valid data from the selected files.")
        self.data = pd.concat(all_dataframes, ignore_index=True)
        req_cols = self.config.get("requested_columns", []); self.available_columns = [c for c in req_cols if c in all_found_columns]; self.missing_columns = [c for c in req_cols if c not in all_found_columns]
        print(f"\n✅ Loading successful. Combined {len(self.data)} rows from {len(self.file_paths)} file(s).");
        if self.missing_columns: print(f"⚠️ Warning: Missing columns: {self.missing_columns}")

    def show_summary(self):
        if self.data is None: return
        for source_file in self.data['source_file'].unique():
            print(f"\n--- Average Temperature Summary for: {source_file} ---"); print(pd.DataFrame(self.data[self.data['source_file'] == source_file][self.available_columns].mean().round(2), columns=['Average Temp (°C)']))
            
    def perform_advanced_analysis(self):
        if self.data is None: return
        control_col, exclude = "70418_T901000_W1T", ["70418_T901100_X1"]
        if control_col not in self.data.columns: print(f"\n⚠️ Advanced Analysis Skipped: Control '{control_col}' not found."); return
        print("\n🔬 Performing Advanced Analysis..."); self.drift_columns = []
        for col in self.available_columns:
            if col != control_col and col not in exclude: self.drift_columns.append(f"Drift_{col}"); self.data[f"Drift_{col}"] = self.data[col] - self.data[control_col]
        if self.drift_columns: print("\n--- Advanced Analysis Summary ---"); print(pd.DataFrame(self.data[self.drift_columns].mean().round(2), columns=['Average Drift from Control (°C)']))

    # --- THE DEFINITIVE AND FINAL LAYOUT FIX FOR LINE PLOTS ---
    def _get_base_layout_settings(self):
        return dict(
            height=700,
            plot_bgcolor='#f0f0f0',
            font=dict(family="Arial, sans-serif", size=14),
            xaxis=dict(title_text="Time", title_font_size=18, showline=True, linecolor='black', mirror=True, showgrid=True, gridwidth=1, gridcolor='white'),
            yaxis=dict(title_text="Temperature (°C)", title_font_size=18, showline=True, linecolor='black', mirror=True, showgrid=True, gridwidth=1, gridcolor='white'),
            legend=dict(title_text='<b style="font-size: 16px;">Thermocouples</b>', bordercolor="Black", borderwidth=1),
            hovermode="x unified",
            hoverlabel=dict(bgcolor="white", bordercolor="black", font_size=14),
            margin=dict(l=100, r=250, t=120, b=80) # Use a large right margin to create space for the legend
        )

    def plot_raw_temperatures(self):
        if self.data is None: return
        print("\nGenerating main temperature plot..."); fig = go.Figure(); layout_settings = self._get_base_layout_settings()
        layout_settings['title'] = dict(text=self.config.get('plot_title'), x=0.5, xanchor='center', font=dict(size=24, family="Arial Black, sans-serif")); fig.update_layout(layout_settings)
        sorted_cols = self.data[self.available_columns].mean().sort_values(ascending=False).index.tolist(); legend_map, dashed_col = self.config.get("legend_map", {}), self.config.get("dashed_column"); self.color_map = {}
        for idx, col in enumerate(sorted_cols):
            color = self.colors[idx % len(self.colors)]; self.color_map[col] = color
            fig.add_trace(go.Scatter(x=self.data["datetime"], y=self.data[col], name=legend_map.get(col, col), mode='lines', line=dict(width=2.5, dash='dash' if col == dashed_col else 'solid', color=color), hovertemplate='<b>%{y:.2f}°C</b><br>%{x|%d-%b-%Y %H:%M:%S}<extra></extra>'))
        fig.show(config=self.config.get("save_config"))

    def plot_drift(self):
        if not hasattr(self, 'drift_columns') or not self.drift_columns: return
        print("\nGenerating drift plot..."); fig = go.Figure(); layout_settings = self._get_base_layout_settings()
        layout_settings['title'] = dict(text="<b>Thermocouple Drift vs. Time</b>", x=0.5, xanchor='center', font=dict(size=24, family="Arial Black, sans-serif")); layout_settings['yaxis']['title_text'] = "Drift from Control (°C)"; fig.update_layout(layout_settings)
        legend_map = self.config.get("legend_map", {}); sorted_cols = self.data[self.drift_columns].abs().mean().sort_values(ascending=False).index.tolist()
        for col in sorted_cols:
            orig_col = col.replace("Drift_", ""); line_color = self.color_map.get(orig_col, 'grey')
            fig.add_trace(go.Scatter(x=self.data["datetime"], y=self.data[col], name=legend_map.get(orig_col, orig_col), mode='lines', line=dict(width=2.5, color=line_color), hovertemplate='<b>Drift: %{y:+.2f}°C</b><br>%{x|%d-%b-%Y %H:%M:%S}<extra></extra>'))
        fig.show(config=self.config.get("save_config"))

    def _prepare_contour_traces(self, mean_temps, config, is_reference=False):
        spatial_map, control_col = config.get("spatial_map"), "70418_T901000_W1T"
        if not spatial_map: return None, None, 0
        avg_control_temp = 0 if is_reference else (mean_temps.get(control_col, 0))
        points, values, hover_texts = [], [], []
        for col, temp in mean_temps.items():
            if col in spatial_map:
                deviation = temp - avg_control_temp; points.append(spatial_map[col]); values.append(deviation)
                hover_texts.append(f"<b>{config['legend_map'].get(col, col)}</b><br>Avg Dev: {deviation:+.2f}°C")
        if len(points) < 3: return None, None, 0
        points, values = np.array(points), np.array(values)
        grid_x, grid_y = np.mgrid[min(points[:,0]):max(points[:,0]):100j, min(points[:,1]):max(points[:,1]):100j]
        grid_z = griddata(points, values, (grid_x, grid_y), method='cubic')
        contour_trace = go.Contour(z=grid_z.T, x=grid_x[:,0], y=grid_y[0,:], colorscale='RdBu_r', colorbar=dict(title='Deviation (°C)'), zmid=0)
        scatter_trace = go.Scatter(x=points[:,0], y=points[:,1], mode='markers', marker=dict(size=12, color='rgba(0,0,0,0.5)', line=dict(width=1, color='White')), text=hover_texts, hoverinfo='text', showlegend=False)
        return contour_trace, scatter_trace, np.nanmax(np.abs(values))

    # --- DEFINITIVELY FIXED: plot_contour now has correct axes and color scale ---
    def plot_contour(self):
        if self.data is None: return
        print("\nGenerating average deviation contour plot...")
        contour_trace, scatter_trace, max_abs_val = self._prepare_contour_traces(self.data[self.available_columns].mean(), self.config)
        if contour_trace:
            fig = go.Figure(data=[contour_trace, scatter_trace])
            fig.update_traces(selector=dict(type='contour'), zmin=-max_abs_val, zmax=max_abs_val)
            fig.update_layout(title="<b>Average Temperature Deviation from Control</b>", height=800, template='plotly_white', showlegend=False)
            fig.update_xaxes(rangemode='tozero', scaleanchor="y", scaleratio=1)
            fig.update_yaxes(rangemode='tozero')
            fig.show(config=self.config.get("save_config"))
        else: print("⚠️ Contour Plot Skipped.")
            
    # --- DEFINITIVELY FIXED: Side-by-side plots are now square with correct axes ---
    def plot_side_by_side_comparison(self, reference_data_dict):
        print("\nGenerating side-by-side contour comparison...")
        fig = make_subplots(rows=1, cols=2, subplot_titles=("<b>Current Run</b>", "<b>Reference Run</b>"))
        trace1, scatter1, max_dev1 = self._prepare_contour_traces(self.data[self.available_columns].mean(), self.config)
        if trace1: fig.add_trace(trace1, row=1, col=1); fig.add_trace(scatter1, row=1, col=1)
        trace2, scatter2, max_dev2 = self._prepare_contour_traces(pd.Series(reference_data_dict), self.config, is_reference=True)
        if trace2: fig.add_trace(trace2, row=1, col=2); fig.add_trace(scatter2, row=1, col=2)
        abs_max = max(max_dev1, max_dev2, 1)
        fig.update_traces(selector=dict(type='contour'), zmin=-abs_max, zmax=abs_max)
        fig.update_xaxes(rangemode='tozero', scaleanchor="y", scaleratio=1)
        fig.update_yaxes(rangemode='tozero', scaleanchor="x", scaleratio=1)
        fig.update_layout(title_text="<b>Side-by-Side Comparison to Reference</b>", height=800)
        fig.show(config=self.config.get("save_config"))

    def plot_dynamic_contour(self, num_frames=75):
        if self.data is None: return
        spatial_map, control_col = self.config.get("spatial_map"), "70418_T901000_W1T"
        if not spatial_map or control_col not in self.data.columns: print("\n⚠️ Dynamic Contour Plot Skipped."); return
        print(f"\nGenerating dynamic contour plot with {num_frames} frames..."); contour_cols = [c for c in self.available_columns if c in spatial_map]
        points = np.array([spatial_map[col] for col in contour_cols]); all_deviations = self.data[contour_cols].sub(self.data[control_col], axis=0)
        max_abs_deviation = all_deviations.abs().max().max();
        if pd.isna(max_abs_deviation) or max_abs_deviation == 0: max_abs_deviation = 1
        print(f"Animation color scale locked to: ±{max_abs_deviation:.2f}°C"); unique_timestamps = self.data['datetime'].unique()
        step = max(1, len(unique_timestamps) // num_frames); animation_timestamps = unique_timestamps[::step]
        grid_x, grid_y = np.mgrid[min(points[:,0]):max(points[:,0]):100j, min(points[:,1]):max(points[:,1]):100j]
        all_z_data = [griddata(points, self.data[self.data['datetime'] == ts][contour_cols].iloc[0].values - self.data[self.data['datetime'] == ts][control_col].iloc[0], (grid_x, grid_y), method='cubic').T for ts in animation_timestamps]
        fig = go.Figure(data=[go.Contour(z=all_z_data[0], x=grid_x[:,0], y=grid_y[0,:], colorscale='RdBu_r', colorbar=dict(title='Deviation (°C)'), zmid=0, zmin=-max_abs_deviation, zmax=max_abs_deviation)])
        frames = [go.Frame(data=[go.Contour(z=z_data)], name=str(ts)) for z_data, ts in zip(all_z_data, animation_timestamps)]
        slider_steps = [{"method": "animate", "args": [[str(ts)], {"frame": {"duration": 100, "redraw": True}, "mode": "immediate"}], "label": pd.to_datetime(ts).strftime('%H:%M:%S')} for ts in animation_timestamps]
        fig.update(frames=frames)
        fig.update_layout(title="<b>Dynamic Temperature Deviation from Control</b>", xaxis=dict(rangemode='tozero', scaleanchor="y", scaleratio=1), yaxis=dict(rangemode='tozero'), height=800, template='plotly_white', updatemenus=[dict(type="buttons", showactive=False, x=0.1, y=0, xanchor="right", yanchor="top", buttons=[dict(label="Play", method="animate", args=[None, {"frame": {"duration": 150, "redraw": True}, "fromcurrent": True, "transition": {"duration": 0}}]), dict(label="Pause", method="animate", args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "transition": {"duration": 0}}])])], sliders=[dict(active=0, steps=slider_steps, x=0.1, len=0.9, xanchor="left", y=-0.1, yanchor="top")])
        fig.show(config=self.config.get("save_config"))

    def export_dynamic_contour_as_gif(self, num_frames=75, export_folder="."): # Faster default
        if not self.imageio_available: print("❌ GIF Export Failed: 'imageio' library is not installed."); return
        if not export_folder: export_folder = "."
        if not os.path.exists(export_folder): os.makedirs(export_folder)
        filename = os.path.join(export_folder, "dynamic_contour_animation.gif")
        print(f"\n🎬 Exporting animation as '{filename}'..."); frame_dir = "temp_gif_frames"
        if not os.path.exists(frame_dir): os.makedirs(frame_dir)
        spatial_map, control_col = self.config.get("spatial_map"), "70418_T901000_W1T"; contour_cols = [c for c in self.available_columns if c in spatial_map]
        points = np.array([spatial_map[c] for c in contour_cols]); all_deviations = self.data[contour_cols].sub(self.data[control_col], axis=0)
        max_abs_deviation = all_deviations.abs().max().max();
        if pd.isna(max_abs_deviation) or max_abs_deviation == 0: max_abs_deviation = 1
        unique_timestamps = self.data['datetime'].unique(); step = max(1, len(unique_timestamps) // num_frames); animation_timestamps = unique_timestamps[::step]
        grid_x, grid_y = np.mgrid[min(points[:,0]):max(points[:,0]):100j, min(points[:,1]):max(points[:,1]):100j]
        image_files = []
        for i, ts in enumerate(animation_timestamps):
            data_at_timestamp = self.data[self.data['datetime'] == ts]
            if data_at_timestamp.empty: continue
            deviations = data_at_timestamp[contour_cols].iloc[0].values - data_at_timestamp[control_col].iloc[0]
            grid_z = griddata(points, deviations, (grid_x, grid_y), method='cubic').T
            fig = go.Figure(data=[go.Contour(z=grid_z, x=grid_x[:,0], y=grid_y[0,:], colorscale='RdBu_r', colorbar=dict(title='Deviation (°C)'), zmid=0, zmin=-max_abs_deviation, zmax=max_abs_deviation)])
            fig.update_layout(title=f"<b>Dynamic Temp Dev ({ts.strftime('%H:%M:%S')})</b>", template='plotly_white', xaxis=dict(scaleanchor="y", scaleratio=1, rangemode='tozero'), yaxis=dict(rangemode='tozero'))
            frame_filename = os.path.join(frame_dir, f"frame_{i:04d}.png"); fig.write_image(frame_filename, scale=2); image_files.append(frame_filename)
        print(f"Stitching {len(image_files)} frames into a GIF...");
        with imageio.get_writer(filename, mode='I', duration=100, loop=0) as writer:
            for image_file in image_files: writer.append_data(imageio.imread(image_file))
        for image_file in image_files: os.remove(image_file)
        os.rmdir(frame_dir)
        print(f"✅ Successfully saved animation to '{filename}'!")

    def run(self, reference_data=None):
        try:
            if self.data is None: self.load_data()
            self.show_summary(); self.perform_advanced_analysis(); self.plot_raw_temperatures(); self.plot_drift(); self.plot_contour()
            if reference_data is not None: self.plot_side_by_side_comparison(reference_data)
            else: print("\nℹ️ Side-by-side comparison skipped.")
            print("\n--- Optional: Dynamic Contour Plot ---"); quality_map = {'1': 1000, '2': 500, '3': 250, '4': 120, '5': 60, 'n': 0}; chosen_frames = 0
            while True:
                print("Select animation quality:"); [print(f"  [{k}] {v[0]} ({v[1]} frames)") for k, v in {'1': ("Michael Jackson", 1000), '2': ("1080p", 500), '3': ("720p", 250), '4': ("480p", 120), '5': ("Happy", 60)}.items()]; print("  [n] No, skip animation.")
                choice = input("Enter your choice [1-5, n]: ").lower().strip()
                if choice in quality_map:
                    chosen_frames = quality_map[choice]
                    if chosen_frames > 0: self.plot_dynamic_contour(num_frames=chosen_frames)
                    else: print("Dynamic contour plot skipped.")
                    break
                else: print("Invalid input.")
            if chosen_frames > 0 and self.imageio_available:
                if input("Would you like to save this animation as a GIF? (This can be slow) [y/n]: ").lower().strip() in ['y', 'yes']:
                    export_folder = self.config.get('export_folder_path')
                    self.export_dynamic_contour_as_gif(export_folder=export_folder)
        except Exception as e: print(f"\n--- An error occurred ---\nError: {e}")

print("✅ ThermocoupleAnalyzer class defined.")

In [ ]:
REFERENCE_DEVIATION_DATA = {
    "70418_T654001_X1": -8.82,  # Kathodenblock
    "70418_T654002_X1": 5.08,  # Cell Holder, Top-Left
    "70418_T654003_X1": 19.28, # Cell Holder, Bottom-Left
    "70418_T654004_X1": -4.42,   # Cell Holder, Middle-Right
    "70418_T654008_X1": 2.38,   # Cell Holder, Top-Right
    "70418_T654009_X1": -13.42,   # Cell Holder, Bottom-Right
    "70418_T901000_X1": 0.06,  # Zell Temperatur
}

print("✅ Reference data for comparison loaded.")

In [ ]:
# SET TO 'True' to run the test, 'False' for normal operation with your files.
RUN_TEST_MODE = False

USER_CONFIG = {
    "requested_columns": ["70418_T654001_X1", "70418_T654002_X1", "70418_T654003_X1", "70418_T654004_X1", "70418_T654008_X1", "70418_T654009_X1", "70418_T901000_W1T", "70418_T901000_X1", "70418_T901100_X1"],
    "legend_map": {"70418_T654001_X1": "Kathodenblock", "70418_T654002_X1": "Cell Holder, Top-Left", "70418_T654003_X1": "Cell Holder, Bottom-Left", "70418_T654004_X1": "Cell Holder, Middle-Right", "70418_T654008_X1": "Cell Holder, Top-Right", "70418_T654009_X1": "Cell Holder, Bottom-Right", "70418_T901000_W1T": "Control Thermocouple", "70418_T901000_X1": "Zell Temperatur", "70418_T901100_X1": "Ofen Temperatur"},
    "spatial_map": {"70418_T654001_X1": (0.5, 0.5), "70418_T654002_X1": (0, 1), "70418_T654003_X1": (0, 0), "70418_T654004_X1": (1, 0.5), "70418_T654008_X1": (1, 1), "70418_T654009_X1": (1, 0), "70418_T901000_X1": (0, 0.5)},
    "dashed_column": "70418_T901000_W1T",
     "plot_title": "<b>Temperature vs. Time</b>",
    
    # --- DEFINITIVE SAVE CONFIGURATION FOR PLOTLY'S "SAVE" BUTTON ---
    "save_config": {'toImageButtonOptions': {'format': 'svg', 'filename': 'custom_plot_export', 'scale': 3}}
}
print("✅ User configuration loaded.")

In [ ]:
try: ref_data = REFERENCE_DEVIATION_DATA
except NameError: ref_data = None

# --- NEW: Graphical Folder Picker at the Start ---
export_path = None
if input("Do you want to select a custom folder to save GIFs? [y/n]: ").lower().strip() in ['y', 'yes']:
    print("Opening folder selection dialog...")
    root = tk.Tk(); root.withdraw(); root.attributes('-topmost', True); root.lift(); root.focus_force()
    export_path = filedialog.askdirectory(parent=root, title="Select a folder for your exports")
    root.destroy()
    if export_path: print(f"✅ Exports will be saved to: {export_path}")
    else: print("⚠️ No folder selected. Exports will be saved in the notebook directory.")
# Add the chosen path to the user config to be used by the analyzer
USER_CONFIG['export_folder_path'] = export_path


if RUN_TEST_MODE:
    while True:
        print("\n--- Select a Test Scenario ---"); print("  [1] Fast Center-Out Ripple"); print("  [2] Interfering Waves (Complex)"); print("  [3] Pulsing Hotspot (Dramatic)"); choice = input("Enter your choice [1-3]: ").strip()
        if choice == '1': test_df = generate_center_out_ripple_data(USER_CONFIG); break
        elif choice == '2': test_df = generate_interfering_waves_data(USER_CONFIG); break
        elif choice == '3': test_df = generate_pulsing_hotspot_data(USER_CONFIG); break
        else: print("Invalid input.")
    analyzer = ThermocoupleAnalyzer(file_paths=["Test Data"], config=USER_CONFIG, imageio_available=IMAGEIO_AVAILABLE)
    analyzer.data, analyzer.available_columns = test_df, [c for c in USER_CONFIG['requested_columns'] if c in test_df.columns]
    analyzer.run(reference_data=ref_data)
else:
    try: from ctypes import windll; windll.shcore.SetProcessDpiAwareness(1)
    except Exception: pass
    root = tk.Tk(); root.withdraw(); root.attributes('-topmost', True); root.lift(); root.focus_force()
    print("Opening file selection dialog..."); file_paths = filedialog.askopenfilenames(parent=root, title="Select One or More Thermocouple Data Files", filetypes=(("Excel files", "*.xlsx *.xls"), ("CSV files", "*.csv"), ("All files", "*.*")))
    root.attributes('-topmost', False); root.destroy()
    if file_paths:
        print(f"{len(file_paths)} file(s) selected.")
        analyzer = ThermocoupleAnalyzer(file_paths, USER_CONFIG, imageio_available=IMAGEIO_AVAILABLE)
        analyzer.run(reference_data=ref_data)
    else: print("No files were selected.")